In [1]:
import gym
import numpy as np
import matplotlib.pyplot as plt
from q_learning import QLearningAgent

# 自动重载外部模块
%load_ext autoreload
%autoreload 2

def rel_error(x, y):
    """计算相对误差，用于结果比对验证"""
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
# 初始化环境
env = gym.make('Taxi-v3')
n_states = env.observation_space.n
n_actions = env.action_space.n

print(f"状态空间大小: {n_states}")
print(f"动作空间大小: {n_actions}")

状态空间大小: 500
动作空间大小: 6


In [ ]:
# 实例化一个用来测试的 agent
test_agent = QLearningAgent(n_states=10, n_actions=4, epsilon=0.0)

# 手动修改部分 Q 值
test_agent.q_table[0] = np.array([1.0, 5.0, 2.0, 0.5])

# 当 epsilon=0 时，它应该 100% 采取最大 Q 值的动作 (即动作 1)
action_greedy = test_agent.choose_action(state=0)
print(f"Greedy 选择的动作 (期望为 1): {action_greedy}")
assert action_greedy == 1, "Epsilon-Greedy 利用 (Exploitation) 部分实现有误！"

# 当 epsilon=1 时，它应该完全随机采取动作
test_agent.epsilon = 1.0
actions_random = [test_agent.choose_action(state=0) for _ in range(1000)]
action_counts = np.bincount(actions_random, minlength=4)
frequencies = action_counts / 1000

print(f"完全随机时各动作选择频率: {frequencies}")
assert np.all(frequencies > 0.2), "Epsilon-Greedy 探索 (Exploration) 部分实现有误！频率分布似乎不均匀。"

print("恭喜！Epsilon-Greedy 策略测试通过！")

In [ ]:
# 实例化测试 agent
test_agent = QLearningAgent(n_states=5, n_actions=3, learning_rate=0.1, discount_factor=0.9, epsilon=0.1)

# 设置初始状态
test_agent.q_table[0] = np.array([0.0, 0.0, 0.0])
test_agent.q_table[1] = np.array([10.0, 20.0, 5.0]) # 下一个状态的最大 Q 值为 20.0

# 模拟一步 transition: 状态0，采取动作1，获得奖励5，转移到状态1
test_agent.update(state=0, action=1, reward=5, next_state=1)

# 手动计算正确答案:
# Q(0,1) = 0.0 + 0.1 * (5 + 0.9 * 20.0 - 0.0) = 0.1 * (5 + 18) = 2.3
expected_q_value = 2.3

actual_q_value = test_agent.q_table[0, 1]
error = rel_error(actual_q_value, expected_q_value)

print(f"期望的 Q 值: {expected_q_value}")
print(f"你的模型更新后的 Q 值: {actual_q_value}")
print(f"相对误差: {error}")

if error < 1e-7:
    print("恭喜！Q-Learning 更新公式测试通过！")
else:
    print("测试未通过，请检查你在 q_learning.py 中的 update 公式。")

In [ ]:
# 初始化训练参数
episodes = 2000
agent = QLearningAgent(n_states, n_actions, learning_rate=0.1, discount_factor=0.99, epsilon=1.0)
epsilon_decay = 0.995
min_epsilon = 0.01

rewards_history = []

for ep in range(episodes):
    state, _ = env.reset()
    done = False
    total_reward = 0
    
    while not done:
        # 1. 选择动作
        action = agent.choose_action(state)
        
        # 2. 与环境交互
        next_state, reward, done, truncated, _ = env.step(action)
        done = done or truncated
        
        # 3. 更新 Q-table
        agent.update(state, action, reward, next_state)
        
        state = next_state
        total_reward += reward
        
    # 记录本回合奖励并衰减 epsilon
    rewards_history.append(total_reward)
    agent.epsilon = max(min_epsilon, agent.epsilon * epsilon_decay)
    
    if (ep + 1) % 200 == 0:
        avg_reward = np.mean(rewards_history[-200:])
        print(f"Episode {ep+1}/{episodes} | Average Reward (last 200): {avg_reward:.2f} | Epsilon: {agent.epsilon:.3f}")

print("训练完成！")

In [ ]:
# 使用滑动窗口平均来平滑曲线
window_size = 100
smoothed_rewards = np.convolve(rewards_history, np.ones(window_size)/window_size, mode='valid')

plt.plot(smoothed_rewards)
plt.title('Q-Learning Training Progress on Taxi-v3')
plt.xlabel('Episodes')
plt.ylabel('Smoothed Total Reward')
plt.grid()
plt.show()